# FAI Football CV — Tracking Pipeline (v0.3)

Turns a short clip of game film into **per-frame player tracking**, a **two-team split**, an optional **top-down field map**, and a **JSON export** in the exact shape the FAI Film Room can import.

This is the football-adapted, lightweight cousin of Roboflow's *Basketball AI* pipeline. It favors choices that actually run today with minimal setup:

| Stage | This notebook (v0.3) | Heavier upgrade (later) |
|---|---|---|
| Detect players | **RF-DETR** — auto-uses your fine-tuned detector if present, else COCO `person` | — |
| Field filter | **Homography bounds** — drops refs / sideline / crowd | Trained field-keypoint model |
| Track | **ByteTrack** | SAM2 segmentation tracking |
| Team split | **Jersey chroma (LAB) + K-means** | SigLIP embeddings + UMAP + K-means |
| Field map | **Click 4 points (homography)** | Trained field-keypoint model |
| Jersey numbers | **EasyOCR, voted across frames per track** | SmolVLM2 fine-tuned OCR |
| Ball tracking | **not included** | — |

### Honest expectations
- Works best on **stable, wide sideline film**. Heavy pan/zoom, end-zone piles, and overlapping bodies degrade tracking — that's the real challenge, so run it as a *proof on your film* before investing more.
- The base detector finds **people**, not football players specifically — but once the homography is set, the field-bounds filter drops refs, sideline, and crowd automatically, and a fine-tuned detector (auto-used) fixes the rest.
- **No ball tracking** — a brown ball under stadium lights is near-impossible to track, and formation/tendency scouting is about where *players* line up and move, not the ball. Intentionally omitted.
- **Night film is the hard case.** Low light + stadium glare wash out jersey colors and add motion blur. This notebook fights that with an optional low-light boost (CLAHE) and a brightness-independent team split, but expect night HS film to be noisier than a daytime clip — run the proof on a real night game.
- No GPU = very slow. Set the Colab runtime to GPU first (next cell).

## 0. Setup
In Colab: **Runtime ▸ Change runtime type ▸ GPU**. Then run the install cell (~1–2 min).

In [ ]:
# One-time install. Pinned loosely; if an API changed, pin versions.
!pip -q install rfdetr supervision opencv-python-headless scikit-learn matplotlib easyocr
import torch; print('CUDA available:', torch.cuda.is_available())

## 1. Configure
Upload a short clip (10–30s is plenty for a proof) via the Colab Files panel, or mount Drive. Then set the path and options below.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    clip_path: str = 'clip.mp4'      # uploaded film clip
    angle: str = 'sideline'          # 'sideline' | 'endzone' (affects your homography points)
    conf: float = 0.4                # detection confidence (start lower for dark night film)
    low_light: bool = True           # CLAHE contrast boost before detection (helps night film)
    frame_stride: int = 3            # process every Nth frame (3 => ~10 fps from 30fps film)
    person_class_id: int = 1         # COCO 'person' id (base model). Fine-tuned 'player' is usually 0 — the detect cell prints the ids.
    max_frames: int = 600            # safety cap so a long upload doesn't run forever
    finetuned_path: str = 'rfdetr_football/checkpoint_best_total.pth'  # auto-used if it exists (see the fine-tune notebook)
    field_margin_yds: float = 8.0    # keep players within this margin of the field; drops refs/sideline/crowd (needs homography)
    read_numbers: bool = True        # OCR jersey numbers per track (EasyOCR), voted across frames
    number_min_box_h: int = 55       # skip boxes shorter than this — too small/far to read
    number_min_conf: float = 0.35    # per-frame OCR confidence floor
    number_min_votes: int = 3        # a track needs this many agreeing reads to get a number
    out_json: str = 'fai_tracking.json'

CFG = Config()
print(CFG)

## 2. Detect players on one frame
Sanity check that detection works and that `person_class_id` is right before processing the whole clip.

In [ ]:
import os, cv2, numpy as np, supervision as sv
from rfdetr import RFDETRBase
from PIL import Image

# Use your fine-tuned football detector automatically once it exists; otherwise
# fall back to the base COCO 'person' model (catches ~half the players on night film).
if os.path.exists(CFG.finetuned_path):
    model = RFDETRBase(pretrain_weights=CFG.finetuned_path)
    print('detector: fine-tuned ->', CFG.finetuned_path, '(set CFG.person_class_id to your player class if needed)')
else:
    model = RFDETRBase()
    print('detector: base COCO person. Train a football detector for all-22 — see football_detector_finetune.ipynb')

cap = cv2.VideoCapture(CFG.clip_path)
assert cap.isOpened(), f'Could not open {CFG.clip_path} — upload a clip and set CFG.clip_path'
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok, frame0 = cap.read(); cap.release()
assert ok, 'Could not read first frame'
print(f'{W}x{H} @ {fps:.1f}fps')

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def enhance(bgr):
    # Low-light boost: lift contrast on the L (lightness) channel only, so
    # dark night frames get brighter without shifting jersey colors.
    if not CFG.low_light: return bgr
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = _clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

rgb = cv2.cvtColor(enhance(frame0), cv2.COLOR_BGR2RGB)
det = model.predict(Image.fromarray(rgb), threshold=CFG.conf)
people = det[det.class_id == CFG.person_class_id]
print('people detected on frame 0:', len(people))

box_ann = sv.BoxAnnotator()
annotated = box_ann.annotate(rgb.copy(), people)
sv.plot_image(annotated, size=(12, 7))

## 3. Detect + track across the clip (ByteTrack)
One record per player per processed frame, a torso-color sample for the team split, and — if `read_numbers` is on — a jersey-number OCR vote. One blurry frame lies, so numbers are **voted across every frame a track is seen**; a track needs `number_min_votes` agreeing reads to get a number.

In [ ]:
tracker = sv.ByteTrack(frame_rate=fps / CFG.frame_stride)

ocr_reader = None
if CFG.read_numbers:
    import easyocr
    ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

def band_crop(img_rgb, box):
    # Jersey band (chest area) of a player box.
    x1, y1, x2, y2 = [int(v) for v in box]
    h = y2 - y1
    ty1, ty2 = y1 + int(0.20 * h), y1 + int(0.55 * h)
    return img_rgb[max(ty1,0):max(ty2,1), max(x1,0):max(x2,1)]

def torso_color(img_rgb, box):
    # Mean LAB chroma (a, b) of the jersey band — brightness-independent, so
    # washed-out night jerseys still separate by hue.
    crop = band_crop(img_rgb, box)
    if crop.size == 0: return np.array([0.0, 0.0])
    lab = cv2.cvtColor(crop, cv2.COLOR_RGB2LAB)
    return lab[:, :, 1:3].reshape(-1, 2).mean(axis=0)   # (a, b)

def read_number(img_rgb, box):
    # Best 1-2 digit token on the jersey this frame, or None. Votes are
    # aggregated per track after the loop.
    if ocr_reader is None or (box[3] - box[1]) < CFG.number_min_box_h:
        return None
    crop = band_crop(img_rgb, box)
    if crop.size == 0: return None
    best = None
    for _bb, txt, conf in ocr_reader.readtext(crop, allowlist='0123456789', detail=1):
        digits = ''.join(ch for ch in txt if ch.isdigit())
        if 1 <= len(digits) <= 2 and conf >= CFG.number_min_conf and (best is None or conf > best[1]):
            best = (digits, conf)
    return best[0] if best else None

records = []          # dicts: frame_idx, t, track_id, box(x1y1x2y2), foot(px)
track_colors = {}     # track_id -> list of torso colors
track_numbers = {}    # track_id -> list of per-frame digit votes

cap = cv2.VideoCapture(CFG.clip_path)
idx = 0; processed = 0
while processed < CFG.max_frames:
    ok, frame = cap.read()
    if not ok: break
    if idx % CFG.frame_stride != 0:
        idx += 1; continue
    rgb = cv2.cvtColor(enhance(frame), cv2.COLOR_BGR2RGB)
    det = model.predict(Image.fromarray(rgb), threshold=CFG.conf)
    det = det[det.class_id == CFG.person_class_id]
    det = tracker.update_with_detections(det)
    t = idx / fps
    for box, tid in zip(det.xyxy, det.tracker_id):
        if tid is None: continue
        tid = int(tid)
        x1, y1, x2, y2 = box
        foot = ((x1 + x2) / 2.0, y2)   # bottom-center = where the player stands
        records.append({'frame_idx': idx, 't': round(float(t), 3), 'track_id': tid,
                        'box': [float(x1), float(y1), float(x2), float(y2)],
                        'foot': [float(foot[0]), float(foot[1])]})
        track_colors.setdefault(tid, []).append(torso_color(rgb, box))
        num = read_number(rgb, box)
        if num is not None: track_numbers.setdefault(tid, []).append(num)
    processed += 1; idx += 1
cap.release()

from collections import Counter
number_of = {}   # track_id -> jersey number (majority vote, min votes)
for tid, votes in track_numbers.items():
    label, count = Counter(votes).most_common(1)[0]
    if count >= CFG.number_min_votes:
        number_of[tid] = label
print(f'processed {processed} frames, {len(records)} observations, {len(track_colors)} tracks, {len(number_of)} numbered')

## 4. Split into two teams (jersey chroma → K-means)
Clusters each *track's* average jersey **chroma** (LAB a/b, brightness removed) into two groups — so night jerseys that look dark still separate by hue. Good enough for contrasting kits; struggles when both teams wear similar colors (upgrade path: SigLIP embeddings).

In [ ]:
from sklearn.cluster import KMeans

tids = [t for t in track_colors if len(track_colors[t]) >= 2]
X = np.array([np.mean(track_colors[t], axis=0) for t in tids])
team_of = {}
if len(tids) >= 2:
    labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
    team_of = {tid: ('A' if lab == 0 else 'B') for tid, lab in zip(tids, labels)}
print('teams assigned:', len(team_of))

## 5. Map to the top-down field (click 4 points)
No coordinate typing. Run the cell, then **click 4 field points on the image** and it builds the homography for you.

**Click order:** near-left, near-right, far-left, far-right — i.e. two yard lines where each meets the near sideline, then where each meets the far sideline (the 4 corners of a box on the field).

Set `G` to how many **yards apart** your two lines are. Field coords are yards (length `x` 0–100, width `y` 0–53.3). Skip this cell to leave `field` null — image-space tracking still exports.

In [ ]:
# Click-to-calibrate: click 4 points on the image, get the homography. No typing coords.
import base64, json
from google.colab.output import eval_js
from IPython.display import display, Javascript

G = 10   # yards between the two yard lines you will click
FIELD_PTS = [(0, 0), (G, 0), (0, 53.3), (G, 53.3)]

_, _buf = cv2.imencode('.png', frame0)
_b64 = base64.b64encode(_buf).decode()

_js_template = '''
async function pick(n) {
  return await new Promise((resolve) => {
    const img = new Image();
    img.onload = () => {
      const info = document.createElement('div');
      info.textContent = 'Click ' + n + ' points: near-left, near-right, far-left, far-right';
      info.style = 'font:16px sans-serif;margin:6px 0;color:#111;background:#c6f24e;padding:4px';
      const c = document.createElement('canvas');
      c.width = img.width; c.height = img.height; c.style.maxWidth = '100%';
      c.getContext('2d').drawImage(img, 0, 0);
      document.body.appendChild(info); document.body.appendChild(c);
      const ctx = c.getContext('2d'); const pts = [];
      c.addEventListener('click', (e) => {
        const r = c.getBoundingClientRect();
        const x = Math.round((e.clientX - r.left) * img.width / r.width);
        const y = Math.round((e.clientY - r.top) * img.height / r.height);
        pts.push([x, y]);
        ctx.fillStyle = 'red'; ctx.beginPath(); ctx.arc(x, y, 9, 0, 7); ctx.fill();
        ctx.fillStyle = 'yellow'; ctx.font = 'bold 28px sans-serif'; ctx.fillText(pts.length, x + 12, y);
        if (pts.length >= n) resolve(JSON.stringify(pts));
      });
    };
    img.src = 'IMG_SRC';
  });
}
'''
_js = _js_template.replace('IMG_SRC', 'data:image/png;base64,' + _b64)
display(Javascript(_js))
IMG_PTS = json.loads(eval_js('pick(4)'))
H, _ = cv2.findHomography(np.array(IMG_PTS, np.float32), np.array(FIELD_PTS, np.float32))
print('clicked:', IMG_PTS)
print('homography ready' if H is not None else 'need 4 points')

def to_field(px, py):
    if H is None: return None
    p = np.array([[[px, py]]], np.float32)
    fx, fy = cv2.perspectiveTransform(p, H)[0][0]
    return [round(float(fx), 2), round(float(fy), 2)]

def on_field(f):
    # With a homography set, keep only players inside the field (+ margin) —
    # this drops refs, sideline players, and crowd. No homography → keep all.
    if f is None: return True
    m = CFG.field_margin_yds
    return -m <= f[0] <= 100 + m and -m <= f[1] <= 53.3 + m

## 6. Preview the top-down radar
Scatter of one frame's players on a field rectangle, colored by team. Only meaningful once the homography is set.

In [ ]:
import matplotlib.pyplot as plt
sample_t = records[len(records)//2]['t'] if records else 0
pts = [r for r in records if abs(r['t'] - sample_t) < 1e-6]
plt.figure(figsize=(11,6))
plt.gca().add_patch(plt.Rectangle((0,0),100,53.3,fill=False))
for yl in range(10,100,10): plt.plot([yl,yl],[0,53.3],color='0.85',lw=1)
for r in pts:
    f = to_field(*r['foot'])
    if f is None or not on_field(f): continue
    c = {'A':'tab:red','B':'tab:blue'}.get(team_of.get(r['track_id']), 'gray')
    plt.scatter(f[0], f[1], c=c, s=60)
plt.xlim(-5,105); plt.ylim(-5,58); plt.title(f'Top-down @ t={sample_t:.2f}s'); plt.show()

## 7. Export JSON for the FAI Film Room
This is the **contract** with the app-side importer: normalized image coords (0–1, matching the Film Room's overlay), optional field yards, team, and track id. One entry per player per frame, grouped by frame.

In [ ]:
from collections import defaultdict
by_t = defaultdict(list)
dropped = 0
for r in records:
    fx, fy = r['foot']
    field = to_field(fx, fy)
    if not on_field(field):   # off-field (ref / sideline / crowd) when a homography is set
        dropped += 1
        continue
    by_t[r['t']].append({
        'trackId': r['track_id'],
        'team': team_of.get(r['track_id']),
        'number': number_of.get(r['track_id']),   # voted jersey number, or null
        'img': {'x': round(fx / W, 4), 'y': round(fy / H, 4)},   # normalized 0-1 to frame
        'field': field,                       # yards or null
    })

out = {
    'meta': {'source': CFG.clip_path, 'fps': fps, 'angle': CFG.angle,
             'frameStride': CFG.frame_stride, 'createdWith': 'fai-football-cv v0.3'},
    'frames': [{'t': t, 'players': by_t[t]} for t in sorted(by_t)],
}
import json
with open(CFG.out_json, 'w') as f: json.dump(out, f)
print('wrote', CFG.out_json, '—', len(out['frames']), 'frames;', dropped, 'off-field detections dropped')
print(json.dumps(out['frames'][0], indent=2)[:600] if out['frames'] else 'no frames')

## 8. Limitations & where this goes next

**What v0.3 added**
- **Jersey-number OCR**: EasyOCR reads the chest band each frame, and the number is **voted per track** across every frame it's seen (`number_min_votes`), so one blurry read can't win. The exported `number` field is filled when a track has enough agreeing reads.

**Earlier (v0.2)**
- **Auto-uses your fine-tuned detector** (`CFG.finetuned_path`) the moment it exists.
- **Field-bounds filtering**: with the homography set, off-field detections (refs, sideline, chain crew, crowd) are dropped from the radar and the export.

**Known limits**
- Jersey OCR is **best-effort on HS night film**: motion blur, turned backs, and 1-vs-2-digit ambiguity mean many tracks stay unnumbered. Raise `number_min_box_h` / `number_min_conf` for fewer-but-surer reads, lower them for more. A fine-tuned jersey model (SmolVLM2) is the accuracy ceiling.
- Base COCO detector catches people, not football players (~half on night film). The fine-tune (auto-used) is the real fix.
- **Night film**: CLAHE + LAB-chroma team split help, but low light/blur still cost detections and swap IDs. If sparse, lower `CFG.conf` and/or raise CLAHE `clipLimit`.
- ByteTrack IDs swap through heavy occlusion (the snap, piles). SAM2 is the upgrade.
- Chroma team split still fails when both teams wear similar colors; SigLIP embeddings fix it.
- No ball tracking (by design — impractical at night and not needed for formation/tendency scouting).

**How it feeds the app**
- `fai_tracking.json` uses the Film Room's normalized 0–1 image coords, so the Football CV importer drops these straight in as **player tracks** — now with team + jersey number — and the field coords can **suggest a formation**, feeding the tendency/scouting engine and the sideline dashboard.

**Next upgrades**
1. SigLIP team classifier (robust team split on similar/washed-out jerseys).
2. ByteTrack → SAM2 for occlusion-robust tracking.
3. Fine-tuned jersey-number model (SmolVLM2) for higher OCR accuracy.